<a href="https://colab.research.google.com/github/mrxy56/Information-Retrieval-Methods/blob/main/BM25_and_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!pip -q install sentence-transformers faiss-cpu rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 205.8 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import faiss

from collections import defaultdict
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.datasets import fetch_20newsgroups

In [3]:
categories = [
    "comp.graphics",
    "comp.os.ms-windows.misc",
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.windows.x",
    "rec.autos",
    "rec.motorcycles",
    "rec.sport.baseball",
    "rec.sport.hockey",
    "sci.space"
]

data = fetch_20newsgroups(
    subset="train",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

documents = []
labels = []

for label in range(10):
    ids = np.where(data.target == label)[0][:100]

    for i in ids:
        documents.append(data.data[i])
        labels.append(label)

rng = np.random.default_rng(42)
order = rng.permutation(len(documents))

documents = [documents[i] for i in order]
labels = [labels[i] for i in order]

doc_ids = [f"doc_{i}" for i in range(len(documents))]

queries = {
    "q1": "computer graphics images rendering visualization",
    "q2": "Microsoft Windows operating system software",
    "q3": "IBM PC hardware computer components",
    "q4": "Apple Macintosh computer hardware",
    "q5": "X Window graphical user interface",
    "q6": "cars automobiles engines driving",
    "q7": "motorcycles bikes riders engines",
    "q8": "baseball players teams games",
    "q9": "ice hockey players teams matches",
    "q10": "space exploration astronomy NASA spacecraft"
}

qrels = {
    f"q{i+1}": {
        doc_ids[j]
        for j, label in enumerate(labels)
        if label == i
    }
    for i in range(10)
}

doc_category = {
    doc_ids[i]: categories[labels[i]]
    for i in range(len(doc_ids))
}

print("Documents:", len(documents))
print("Queries:", len(queries))

Documents: 1000
Queries: 10


In [4]:
tokenized_documents = [
    document.lower().split()
    for document in documents
]

bm25 = BM25Okapi(tokenized_documents)

bm25_results = {}

for qid, query in queries.items():
    scores = bm25.get_scores(query.lower().split())
    ranking = np.argsort(scores)[::-1][:10]

    bm25_results[qid] = [
        doc_ids[i]
        for i in ranking
    ]

In [5]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

doc_embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

query_embeddings = model.encode(
    list(queries.values()),
    normalize_embeddings=True
).astype("float32")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [6]:
index = faiss.IndexFlatIP(
    doc_embeddings.shape[1]
)

index.add(doc_embeddings)

scores, indices = index.search(
    query_embeddings,
    10
)

dense_results = {
    qid: [
        doc_ids[i]
        for i in indices[row]
    ]
    for row, qid in enumerate(queries)
}

print("Embedding dimension:", doc_embeddings.shape[1])
print("Vectors in FAISS:", index.ntotal)

Embedding dimension: 384
Vectors in FAISS: 1000


In [7]:
def rrf(rankings, k=60):
    scores = defaultdict(float)

    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1 / (k + rank)

    return [
        doc_id
        for doc_id, score in sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:10]
    ]

hybrid_results = {
    qid: rrf([
        bm25_results[qid],
        dense_results[qid]
    ])
    for qid in queries
}

In [8]:
def precision_at_5(ranking, relevant):
    return sum(
        doc_id in relevant
        for doc_id in ranking[:5]
    ) / 5

def recall_at_5(ranking, relevant):
    return sum(
        doc_id in relevant
        for doc_id in ranking[:5]
    ) / len(relevant)

def mrr(ranking, relevant):
    for rank, doc_id in enumerate(ranking, start=1):
        if doc_id in relevant:
            return 1 / rank

    return 0

In [9]:
def evaluate(results, name):
    rows = []

    for qid in queries:
        rows.append({
            "P@5": precision_at_5(
                results[qid],
                qrels[qid]
            ),
            "Recall@5": recall_at_5(
                results[qid],
                qrels[qid]
            ),
            "MRR": mrr(
                results[qid],
                qrels[qid]
            )
        })

    averages = pd.DataFrame(rows).mean()

    return {
        "Retriever": name,
        "P@5": averages["P@5"],
        "Recall@5": averages["Recall@5"],
        "MRR": averages["MRR"]
    }

results_table = pd.DataFrame([
    evaluate(bm25_results, "BM25"),
    evaluate(dense_results, "Dense"),
    evaluate(hybrid_results, "Hybrid")
])

results_table

,Retriever,P@5,Recall@5,MRR
0,BM25,0.72,0.036,0.733333
1,Dense,0.94,0.047,1.000000
2,Hybrid,0.82,0.041,0.950000


In [10]:
comparison = []

for qid in ["q1", "q6", "q10"]:
    comparison.append({
        "Query": queries[qid],
        "BM25": bm25_results[qid][:5],
        "Dense": dense_results[qid][:5],
        "Hybrid": hybrid_results[qid][:5]
    })

pd.DataFrame(comparison)

,Query,BM25,Dense,Hybrid
0,computer graphics images rendering visualization,"[doc_312, doc_857, doc_735, doc_503, doc_850]","[doc_646, doc_947, doc_391, doc_275, doc_178]","[doc_156, doc_312, doc_646, doc_857, doc_947]"
1,cars automobiles engines driving,"[doc_751, doc_810, doc_472, doc_871, doc_10]","[doc_15, doc_285, doc_255, doc_273, doc_716]","[doc_810, doc_751, doc_15, doc_285, doc_472]"
2,space exploration astronomy NASA spacecraft,"[doc_110, doc_840, doc_584, doc_944, doc_989]","[doc_944, doc_508, doc_795, doc_989, doc_584]","[doc_944, doc_584, doc_840, doc_989, doc_795]"


In [11]:
qid = "q10"

result_table = pd.DataFrame({
    "Rank": range(1, 11),
    "Document ID": hybrid_results[qid],
    "Retrieved Category": [
        doc_category[doc_id]
        for doc_id in hybrid_results[qid]
    ],
    "Relevant": [
        doc_id in qrels[qid]
        for doc_id in hybrid_results[qid]
    ]
})

result_table

,Rank,Document ID,Retrieved Category,Relevant
0,1,doc_944,sci.space,True
1,2,doc_584,sci.space,True
2,3,doc_840,sci.space,True
3,4,doc_989,sci.space,True
4,5,doc_795,sci.space,True
5,6,doc_110,sci.space,True
6,7,doc_508,sci.space,True
7,8,doc_426,sci.space,True
8,9,doc_728,sci.space,True
9,10,doc_104,sci.space,True


In [12]:
qid = "q10"

top_docs = hybrid_results[qid][:3]

context = "\n\n".join(
    documents[int(doc_id.split("_")[1])]
    for doc_id in top_docs
)

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{queries[qid]}

Answer:
"""

print(prompt)


Answer the question using only the context below.

Context:
Archive-name: space/new_probes
Last-modified: $Date: 93/04/01 14:39:17 $

UPCOMING PLANETARY PROBES - MISSIONS AND SCHEDULES

    Information on upcoming or currently active missions not mentioned below
    would be welcome. Sources: NASA fact sheets, Cassini Mission Design
    team, ISAS/NASDA launch schedules, press kits.


    ASUKA (ASTRO-D) - ISAS (Japan) X-ray astronomy satellite, launched into
    Earth orbit on 2/20/93. Equipped with large-area wide-wavelength (1-20
    Angstrom) X-ray telescope, X-ray CCD cameras, and imaging gas
    scintillation proportional counters.


    CASSINI - Saturn orbiter and Titan atmosphere probe. Cassini is a joint
    NASA/ESA project designed to accomplish an exploration of the Saturnian
    system with its Cassini Saturn Orbiter and Huygens Titan Probe. Cassini
    is scheduled for launch aboard a Titan IV/Centaur in October of 1997.
    After gravity assists of Venus, Earth and Jup

In [13]:
results_table.to_csv(
    "retrieval_results.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.
